In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

DATA_DIR   = Path("C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed")
INPUT_FILE = DATA_DIR / "final_dataset.parquet"
OUT_FILE   = DATA_DIR / "modelling_panel.parquet"

print("Loading data …")
df = pd.read_parquet(INPUT_FILE)
df["Date"]       = pd.to_datetime(df["Date"])
df["YearMonth"]  = df["Date"].dt.to_period("M")


Loading data …


In [2]:
# Clean weather columns 
weather_cols = ["DR", "RH", "SQ", "TG", "TN", "TX", "RHX", "FHVEC", "VVX", "T10N"]
for col in weather_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.strip(), errors="coerce"
        )

# Align cancellations that have no disruption_count entry
if {"Final arrival cancelled", "Completely cancelled", "Disruption Count"}.issubset(df.columns):
    mask = (
        (df["Final arrival cancelled"].astype(int) > 0) &
        (df["Completely cancelled"].astype(int)    > 0) &
        (df["Disruption Count"]                    == 0)
    )
    df.loc[mask, "Disrupted"]        = True
    df.loc[mask, "Disruption Count"] = df.loc[mask, "Final arrival cancelled"]

df["Disrupted"] = df["Disrupted"].astype(int)



In [3]:
# Any column that starts with source_ or target_ and is not the key itself
ses_cols = [
    c for c in df.columns
    if c.startswith(("source_", "target_")) and c not in ("source", "target")
]

# Aggregate to monthly link-level panel

agg_ops = {
    "Disrupted":     "max",  
    "Rides planned": "sum",   
}
for col in weather_cols:
    if col in df.columns:
        agg_ops[col] = "mean"
for col in ses_cols:
    if col in df.columns:
        agg_ops[col] = "mean"

panel = (
    df.groupby(["source", "target", "YearMonth"])
      .agg(agg_ops)
      .reset_index()
      .rename(columns={"Rides planned": "rides_planned"})
)

# Number of active service days in the month 
stop_counts = (
    df.groupby(["source", "target", "YearMonth"])["Date"]
      .nunique()
      .reset_index()
      .rename(columns={"Date": "stop_count"})
)
panel = panel.merge(stop_counts, on=["source", "target", "YearMonth"], how="left")


In [4]:
# Remoteness Index 
for prefix in ("source_", "target_"):
    hw  = f"{prefix}Dist_Highway_Entrance"
    mts = f"{prefix}Dist_Major_Transfer_Station"
    if hw in panel.columns and mts in panel.columns:
        panel[f"{prefix}Remoteness_Index"] = (panel[hw] + panel[mts]) / 2

# Lag features 
panel = panel.sort_values(["source", "target", "YearMonth"]).reset_index(drop=True)

# Lagged disruption status (safe: previous months only)
for lag in [1, 2, 3]:
    panel[f"disrupted_lag{lag}"] = (
        panel.groupby(["source", "target"])["Disrupted"].shift(lag)
    )

# Rolling historical delay frequency — window is ENTIRELY in the past.
# shift(1) ensures month t is never included in its own rolling window.
panel["delay_freq_3m"] = (
    panel.groupby(["source", "target"])["Disrupted"]
         .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
panel["delay_freq_6m"] = (
    panel.groupby(["source", "target"])["Disrupted"]
         .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)


In [5]:
# Topology features

# Aggregate rides_planned per (source, target) across the full panel
edge_weights = (
    panel.groupby(["source", "target"])["rides_planned"]
         .sum()
         .reset_index()
         .rename(columns={"rides_planned": "total_planned"})
)

G_static = nx.Graph()
for _, row in edge_weights.iterrows():
    G_static.add_edge(
        row["source"], row["target"],
        weight=max(row["total_planned"], 1)
    )


# Compute centrality measures once on the static graph
degree_c      = dict(G_static.degree())
betweenness_n = nx.betweenness_centrality(G_static, normalized=True, weight="weight")
closeness     = nx.closeness_centrality(G_static, distance="weight")
clustering    = nx.clustering(G_static, weight="weight")
edge_btwn     = nx.edge_betweenness_centrality(G_static, normalized=True, weight="weight")

try:
    eigenvector = nx.eigenvector_centrality(G_static, max_iter=1000, weight="weight")
except nx.PowerIterationFailedConvergence:
    print("Warning: eigenvector centrality did not converge — defaulting to 0.")
    eigenvector = {n: 0.0 for n in G_static.nodes()}

def get_topo_row(src, tgt):
    """Return a dict of static topology features for one (source, target) pair."""
    edge_key   = (src, tgt) if G_static.has_edge(src, tgt) else (tgt, src)
    common_nbr = (
        len(list(nx.common_neighbors(G_static, src, tgt)))
        if G_static.has_node(src) and G_static.has_node(tgt) else 0
    )
    return {
        "topo_src_degree":        degree_c.get(src, 0),
        "topo_tgt_degree":        degree_c.get(tgt, 0),
        "topo_src_betweenness":   betweenness_n.get(src, 0),
        "topo_tgt_betweenness":   betweenness_n.get(tgt, 0),
        "topo_src_closeness":     closeness.get(src, 0),
        "topo_tgt_closeness":     closeness.get(tgt, 0),
        "topo_src_clustering":    clustering.get(src, 0),
        "topo_tgt_clustering":    clustering.get(tgt, 0),
        "topo_src_eigenvector":   eigenvector.get(src, 0),
        "topo_tgt_eigenvector":   eigenvector.get(tgt, 0),
        "topo_edge_betweenness":  edge_btwn.get(edge_key, 0),
        "topo_common_neighbours": common_nbr,
    }

topo_rows = panel.apply(
    lambda r: get_topo_row(r["source"], r["target"]), axis=1
)
topo_df = pd.DataFrame(topo_rows.tolist(), index=panel.index)
panel   = pd.concat([panel, topo_df], axis=1)


In [6]:

# Integer time_id (needed by Tier 2 & 3) 
periods    = sorted(panel["YearMonth"].unique())
period_map = {p: i for i, p in enumerate(periods)}
panel["time_id"] = panel["YearMonth"].map(period_map)


panel = panel.fillna(0)

# Save 
panel.to_parquet(OUT_FILE, index=False)
print({panel.shape})

{(51877, 84)}
